<a href="https://colab.research.google.com/github/szmygielski/lecture1/blob/main/5PUM_regresja_30_03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge, Lasso, HuberRegressor, RANSACRegressor, TheilSenRegressor, SGDRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, explained_variance_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import PredictionErrorDisplay

In [4]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

file_path = "Salary Data.csv"

df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "rkiattisak/salaly-prediction-for-beginer",
  file_path,
)

print("Pierwsze 5 rekordów:", df.head())

/tmp/ipykernel_10099/855972280.py:6: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


Using Colab cache for faster access to the 'salaly-prediction-for-beginer' dataset.
Pierwsze 5 rekordów:     Age  Gender Education Level          Job Title  Years of Experience  \
0  32.0    Male      Bachelor's  Software Engineer                  5.0   
1  28.0  Female        Master's       Data Analyst                  3.0   
2  45.0    Male             PhD     Senior Manager                 15.0   
3  36.0  Female      Bachelor's    Sales Associate                  7.0   
4  52.0    Male        Master's           Director                 20.0   

     Salary  
0   90000.0  
1   65000.0  
2  150000.0  
3   60000.0  
4  200000.0  


In [5]:
df.info()

le = LabelEncoder()

for column in ['Gender', 'Education Level', 'Job Title']:
  if column in df.columns:
    df[column] = le.fit_transform(df[column])

print(df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 375 entries, 0 to 374
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Age                  373 non-null    float64
 1   Gender               373 non-null    object 
 2   Education Level      373 non-null    object 
 3   Job Title            373 non-null    object 
 4   Years of Experience  373 non-null    float64
 5   Salary               373 non-null    float64
dtypes: float64(3), object(3)
memory usage: 17.7+ KB
    Age  Gender  Education Level  Job Title  Years of Experience    Salary
0  32.0       1                0        159                  5.0   90000.0
1  28.0       0                1         17                  3.0   65000.0
2  45.0       1                2        130                 15.0  150000.0
3  36.0       0                0        101                  7.0   60000.0
4  52.0       1                1         22                 20.0  20

Obliczanie VIF

In [56]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

df_clean = df.dropna().copy()
X = df_clean.drop(columns=['Salary'])
y = df_clean['Salary']

def calculate_vif(X_input):
    vif = pd.DataFrame()
    vif["Cecha"] = X_input.columns
    vif["VIF"] = [variance_inflation_factor(X_input.values, i) for i in range(len(X_input.columns))]
    return vif

while True:
    vif_df = calculate_vif(X)
    max_vif = vif_df['VIF'].max()
    if max_vif > 10:
        feature_to_drop = vif_df.sort_values('VIF', ascending=False)['Cecha'].iloc[0]
        X = X.drop(columns=[feature_to_drop])
    else:
        break

print("Ostateczne współczynniki VIF:")
display(vif_df)

Ostateczne współczynniki VIF:


,Cecha,VIF
0,Gender,1.819647
1,Education Level,2.379329
2,Job Title,2.798903
3,Years of Experience,4.147387


Regresja liniowa lasso

In [58]:
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.neighbors import KNeighborsRegressor

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

wyniki = []

lasso = Lasso(alpha=1.0)
lasso.fit(X_train_scaled, y_train)
pred_lasso = lasso.predict(X_test_scaled)
wyniki.append({'Model': 'LASSO', 'MSE': mean_squared_error(y_test, pred_lasso), 'R2': r2_score(y_test, pred_lasso)})

display(pd.DataFrame(wyniki))

,Model,MSE,R2
0,LASSO,2.247585e+08,0.906256


Regresja wielowymiarowa stopnia 2 i 3

In [59]:
for deg in [2, 3]:
    poly = PolynomialFeatures(degree=deg)
    X_poly_train = poly.fit_transform(X_train_scaled)
    X_poly_test = poly.transform(X_test_scaled)

    model_poly = LinearRegression()
    model_poly.fit(X_poly_train, y_train)
    pred_poly = model_poly.predict(X_poly_test)
    wyniki.append({'Model': f'Wielomianowa st. {deg}', 'MSE': mean_squared_error(y_test, pred_poly), 'R2': r2_score(y_test, pred_poly)})

Regresja k-NN i wyświetlenie wyników

In [60]:
knn = KNeighborsRegressor(n_neighbors=5)
knn.fit(X_train_scaled, y_train)
pred_knn = knn.predict(X_test_scaled)
wyniki.append({'Model': 'k-NN', 'MSE': mean_squared_error(y_test, pred_knn), 'R2': r2_score(y_test, pred_knn)})

porownanie_df = pd.DataFrame(wyniki).sort_values('MSE')
display(porownanie_df)

,Model,MSE,R2
0,LASSO,2.247585e+08,0.906256
1,Wielomianowa st. 2,2.647092e+08,0.889593
2,Wielomianowa st. 3,3.324142e+08,0.861354
3,k-NN,3.359545e+08,0.859878
